# LLM router that decide which method to use for retrieval

In [1]:
from pathlib import Path
import re
import pickle
import os
from pprint import pprint

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexMacroNode


In [2]:
load_dotenv()

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INDEX_DIR = PROJECT_ROOT / "storage" / "faiss_index"
CHUNKS_PATH = PROJECT_ROOT / "storage" / "chunks.pkl"
PAPER_DIR   = PROJECT_ROOT / "paper"

EMBEDDING_MODEL = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = FAISS.load_local(
    str(INDEX_DIR),
    embeddings,
    allow_dangerous_deserialization=True,
)

with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)

print("Loaded vector store and", len(chunks), "chunks")

Loaded vector store and 205 chunks


In [3]:
EXCLUDED_DIR_NAMES = {
    "Images",
    "images",
    "numerical_images",
    "old_writeups",
    ".git",
    "having the graph.tex",
    "__pycache__",
}

def should_skip_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)

tex_files = sorted(
    path for path in PAPER_DIR.rglob("*.tex")
    if path.is_file() and not should_skip_path(path)
)

print(f"Found {len(tex_files)} .tex files:\n")
for path in tex_files:
    print("-", path.relative_to(PAPER_DIR))

Found 2 .tex files:

- Appendix.tex
- main.tex


In [4]:
def _arg_text(arg_node):
    if arg_node is None:
        return None
    return "".join(
        n.chars for n in arg_node.nodelist if hasattr(n, "chars")
    ).strip()

In [5]:
NEWTHEOREM_RE = re.compile(
    r"\\newtheorem\{(\w+)\}"   
    r"(?:\[(\w+)\])?"          
    r"\{([^}]+)\}"             
    r"(?:\[(\w+)\])?"          
)

def parse_newtheorem_declarations(tex_files):
    """
    Returns a dict keyed by environment name:
    {
        "theorem":     {"display": "Theorem",     "counter": "theorem",     "parent": None},
        "proposition": {"display": "Proposition", "counter": "proposition", "parent": None},
        "lemma":       {"display": "Lemma",       "counter": "lemma",       "parent": None},
        "corollary":   {"display": "Corollary",   "counter": "corollary",   "parent": None},
    }

     """
    theorems = {}
    for path in tex_files:
        text = path.read_text(encoding="utf-8", errors="ignore")
        for m in NEWTHEOREM_RE.finditer(text):
            envname, shared_counter, display, parent_counter = m.groups()
            theorems[envname] = {
                "display": display.strip(),
                "counter": shared_counter if shared_counter else envname,
                "parent":  parent_counter,
            }
    return theorems


In [6]:

theorems = parse_newtheorem_declarations(tex_files)

print("Theorem-like environments found:")
for env, info in theorems.items():
    shared = (
        f" (shares counter with '{info['counter']}')"
        if info["counter"] != env else ""
    )
    parent = f" (resets per \\{info['parent']})" if info["parent"] else ""
    print(f"  \\begin{{{env}}} → '{info['display']}'{shared}{parent}")

Theorem-like environments found:
  \begin{theorem} → 'Theorem'
  \begin{proposition} → 'Proposition'
  \begin{lemma} → 'Lemma'
  \begin{corollary} → 'Corollary'


In [7]:
INPUT_RE = re.compile(r"\\(?:input|include)\{([^}]+)\}")

def resolve_input_order(tex_files):
    """
    Fancy way of following input include from main.tex to reconstruct the order, to make a counter for all theorems
    
    """
    file_map = {}
    for f in tex_files:
        file_map[f.stem] = f
        file_map[f.name] = f

    main = file_map.get("main") or file_map.get("main.tex")
    if main is None:
        print("WARNING: no main.tex found — using original file order (numbers may be wrong)")
        return tex_files

    ordered = [main]
    text = main.read_text(encoding="utf-8", errors="ignore")

    for m in INPUT_RE.finditer(text):
        name = Path(m.group(1)).stem
        if name in file_map:
            f = file_map[name]
            if f not in ordered:
                ordered.append(f)
        else:
            print(f"WARNING: \\input{{{name}}} found but {name}.tex not in tex_files — skipping")

    unreferenced = [f for f in tex_files if f not in ordered]
    if unreferenced:
        print(f"WARNING: {[f.name for f in unreferenced]} not referenced via \\input — appended at end")
        ordered.extend(unreferenced)

    return ordered


In [8]:

ordered_tex_files = resolve_input_order(tex_files)
print("Processing order:")
for f in ordered_tex_files:
    print(" ", f.name)

Processing order:
  main.tex
  Appendix.tex


Such a tedious work to parse LaTeX files, god bless claude.

In [9]:
def _walk_numbered(nodes, labels, refs, source, theorems, state):
    for node in nodes:

        # ── ENVIRONMENTS ──────────────────────────────────────────────
        if isinstance(node, LatexEnvironmentNode):
            env = node.environmentname.rstrip("*")
            state["env_stack"].append(env)

            if env in theorems:
                counter_name = theorems[env]["counter"]
                parent       = theorems[env]["parent"]

                state["counters"][counter_name] = (
                    state["counters"].get(counter_name, 0) + 1
                )
                num          = state["counters"][counter_name]
                display_name = theorems[env]["display"]

                if state["in_appendix"]:
                    letter = chr(ord("A") + state["appendix_section_index"])
                    if state["appendix_section_numbered"]:
                        display_num = f"{letter}.{num}"
                    elif parent == "section":
                        display_num = f"{letter}.{num}"
                    else:
                        display_num = str(num)
                elif parent == "section":
                    display_num = f"{state['section_num']}.{num}"
                else:
                    display_num = str(num)

                state["pending_display"] = f"{display_name} {display_num}"
                state["pending_num"]     = display_num
                state["pending_env"]     = env

            _walk_numbered(node.nodelist, labels, refs, source, theorems, state)
            state["env_stack"].pop()

            if env in theorems:
                state["pending_display"] = None
                state["pending_num"]     = None
                state["pending_env"]     = None

        # ── MACROS ────────────────────────────────────────────────────
        elif isinstance(node, LatexMacroNode):
            name = node.macroname
            args = (node.nodeargd.argnlist if node.nodeargd else []) or []

            # \appendix
            # Does NOT reset theorem counters on its own. Only flips the flag.
            # Any resets will show up as explicit \setcounter calls below.
            if name == "appendix":
                state["in_appendix"]              = True
                state["appendix_section_index"]   = -1
                state["warnings"].append(
                    f"\\appendix found in {Path(source).name} — "
                    f"section counter switches to letters; "
                    f"theorem counters are NOT reset (LaTeX default). "
                    f"Check for \\setcounter or \\renewcommand calls nearby."
                )

            # \section
            elif name == "section":
                if state["in_appendix"]:
                    state["appendix_section_index"] += 1
                    # Reset theorem counters per section only if appendix
                    # section-style numbering was detected (A.1, A.2, B.1 ...)
                    if state["appendix_section_numbered"]:
                        for info in theorems.values():
                            state["counters"][info["counter"]] = 0
                else:
                    state["section_num"] += 1
                    # Reset section-parented counters in main text
                    for info in theorems.values():
                        if info["parent"] == "section":
                            state["counters"][info["counter"]] = 0

            elif name in ("subsection", "subsubsection"):
                pass

            # \setcounter{name}{value}
            elif name == "setcounter" and len(args) >= 2:
                counter_name = _arg_text(args[0])
                try:
                    value = int(_arg_text(args[1]))
                    old   = state["counters"].get(counter_name, "unknown")
                    state["counters"][counter_name] = value
                    # If we are in the appendix and a theorem counter is
                    # reset to 0, this confirms A.1 style numbering is active.
                    if (
                        state["in_appendix"]
                        and value == 0
                        and counter_name in {info["counter"] for info in theorems.values()}
                    ):
                        state["appendix_section_numbered"] = True
                    state["warnings"].append(
                        f"\\setcounter{{{counter_name}}}{{{value}}} in {Path(source).name} "
                        f"(was {old}) — numbering is non-sequential from this point"
                    )
                except (ValueError, TypeError):
                    pass

            # \addtocounter{name}{delta}
            elif name == "addtocounter" and len(args) >= 2:
                counter_name = _arg_text(args[0])
                try:
                    delta = int(_arg_text(args[1]))
                    old   = state["counters"].get(counter_name, 0)
                    state["counters"][counter_name] = old + delta
                    state["warnings"].append(
                        f"\\addtocounter{{{counter_name}}}{{{delta}}} in {Path(source).name} "
                        f"(was {old}, now {old + delta})"
                    )
                except (ValueError, TypeError):
                    pass

            # \renewcommand{\thelemma}{\thesection.\arabic{lemma}} etc.
            # If after \appendix a \renewcommand references \thesection,
            # the paper switched to A.1 style numbering.
            elif name == "renewcommand" and len(args) >= 2 and state["in_appendix"]:
                cmd_name = _arg_text(args[0]) or ""
                cmd_body = _arg_text(args[1]) or ""
                if "thesection" in cmd_body:
                    state["appendix_section_numbered"] = True
                    state["warnings"].append(
                        f"\\renewcommand{{{cmd_name}}} references \\thesection "
                        f"in {Path(source).name} — confirmed A.1 style appendix numbering"
                    )

            # \label{key}
            elif name == "label" and args:
                key = _arg_text(args[0])
                if key:
                    env   = state["env_stack"][-1] if state["env_stack"] else "section"
                    entry = {
                        "source":      source,
                        "environment": env,
                        "in_appendix": state["in_appendix"],
                    }
                    if state["pending_display"]:
                        entry["display"] = state["pending_display"]
                        entry["number"]  = state["pending_num"]
                        entry["label"]   = key
                    labels[key] = entry

            # \ref, \eqref, etc.
            elif name in ("ref", "eqref", "autoref", "Cref", "cref") and args:
                key = _arg_text(args[0])
                if key:
                    refs.append({
                        "source":   source,
                        "ref_key":  key,
                        "ref_type": name,
                    })

In [10]:
def build_numbered_label_graph(tex_files, theorems):
    labels, refs = {}, []
    state = {
        "counters":                  {info["counter"]: 0 for info in theorems.values()},
        "env_stack":                 [],
        "section_num":               0,
        "in_appendix":               False,
        "appendix_section_index":    -1,
        "appendix_section_numbered": False,   # flipped by \setcounter or \renewcommand
        "pending_display":           None,
        "pending_num":               None,
        "pending_env":               None,
        "warnings":                  [],
    }

    ordered = resolve_input_order(tex_files)
    for path in ordered:
        text  = path.read_text(encoding="utf-8", errors="ignore")
        nodes, _, _ = LatexWalker(text).get_latex_nodes()
        _walk_numbered(nodes, labels, refs, str(path), theorems, state)

    display_to_label = {}
    for key, info in labels.items():
        if "display" in info:
            display_to_label[info["display"]] = key

    return {
        "labels":           labels,
        "refs":             refs,
        "display_to_label": display_to_label,
        "warnings":         state["warnings"],
    }

In [11]:

graph = build_numbered_label_graph(tex_files, theorems)
print(f"{len(graph['labels'])} labels, {len(graph['refs'])} refs")
print(f"{len(graph['display_to_label'])} display-name mappings built")

54 labels, 104 refs
13 display-name mappings built


In [12]:
graph.keys()

dict_keys(['labels', 'refs', 'display_to_label', 'warnings'])

In [13]:
graph['warnings']

['\\appendix found in main.tex — section counter switches to letters; theorem counters are NOT reset (LaTeX default). Check for \\setcounter or \\renewcommand calls nearby.']

In [14]:
graph['display_to_label']

{'Proposition 1': 'eq:mu_bounds',
 'Theorem 1': 't:unfairness_intervals',
 'Theorem 2': 't:fairness_condition',
 'Corollary 1': 'c:fair_regions_params',
 'Theorem 3': 't:fairness_Unaware_DM',
 'Corollary 2': 'c:maximum_unfairness_Unaware',
 'Theorem 4': 't:comparison_of_fairness',
 'Theorem 5': 't:accuracy_full',
 'Lemma 1': 'app:lemma_lin',
 'Lemma 2': 'app_mu_rel_10',
 'Lemma 3': 'app:lemma_signal_divided',
 'Lemma 4': 'app:lemma_six_thresholds',
 'Lemma 5': 'app:lemma_indeterminate'}

In [15]:
graph['labels']['t:unfairness_intervals']

{'source': 'c:\\Tutorial\\rag\\RagTeX\\paper\\main.tex',
 'environment': 'theorem',
 'in_appendix': False,
 'display': 'Theorem 1',
 'number': '1',
 'label': 't:unfairness_intervals'}

Once again, god bless claude for this. Full transparency, it is claude execution, and my plan.

In [16]:
if graph["warnings"]:
    print("WARNINGS (check these against your PDF):")
    for w in graph["warnings"]:
        print(" !", w)
else:
    print("No counter resets or manipulations found.")

print()
print("Display name → label key mappings:")
print(f"  {'Display':30s}  {'Label key':35s}  {'In appendix'}")
print(f"  {'-'*30}  {'-'*35}  {'-'*11}")
for display, key in sorted(graph["display_to_label"].items()):
    info        = graph["labels"][key]
    in_appendix = "YES" if info.get("in_appendix") else ""
    print(f"  {display:30s}  {key:35s}  {in_appendix}")

WARNINGS (check these against your PDF):
 ! \appendix found in main.tex — section counter switches to letters; theorem counters are NOT reset (LaTeX default). Check for \setcounter or \renewcommand calls nearby.

Display name → label key mappings:
  Display                         Label key                            In appendix
  ------------------------------  -----------------------------------  -----------
  Corollary 1                     c:fair_regions_params                
  Corollary 2                     c:maximum_unfairness_Unaware         
  Lemma 1                         app:lemma_lin                        YES
  Lemma 2                         app_mu_rel_10                        YES
  Lemma 3                         app:lemma_signal_divided             YES
  Lemma 4                         app:lemma_six_thresholds             YES
  Lemma 5                         app:lemma_indeterminate              YES
  Proposition 1                   eq:mu_bounds                     

In [17]:
LABEL_RE = re.compile(r"\\label\{([^}]+)\}")
REF_RE   = re.compile(r"\\(?:ref|eqref|autoref|[Cc]ref)\{([^}]+)\}")

def attach_labels_to_chunks(chunks, graph):
    for chunk in chunks:
        own_labels = LABEL_RE.findall(chunk.page_content)
        own_refs   = REF_RE.findall(chunk.page_content)

        chunk.metadata["labels"] = own_labels
        chunk.metadata["refs"]   = own_refs

        # For each ref in this chunk, look up what it points to
        chunk.metadata["ref_targets"] = [
            {
                "key":     k,
                "env":     graph["labels"].get(k, {}).get("environment"),
                "display": graph["labels"].get(k, {}).get("display"),
            }
            for k in own_refs
        ]

        # If this chunk owns a label that has a display name, surface it
        # so retrievers can match "Theorem 2" directly to this chunk
        chunk.metadata["displays"] = [
            graph["labels"][lbl]["display"]
            for lbl in own_labels
            if lbl in graph["labels"] and "display" in graph["labels"][lbl]
        ]



In [18]:
attach_labels_to_chunks(chunks, graph)

# Spot-check: find a chunk that owns a numbered result
for chunk in chunks:
    if chunk.metadata.get("displays"):
        print("chunk_id:", chunk.metadata.get("chunk_id"))
        print("  labels:",    chunk.metadata["labels"])
        print("  displays:",  chunk.metadata["displays"])
        print("  refs:",      chunk.metadata["refs"])
        print("  ref_targets:", chunk.metadata["ref_targets"])
        print("  preview:", chunk.page_content[:200])
        break

chunk_id: 2
  labels: ['app:lemma_lin', 'app:lemma_mu_relations']
  displays: ['Lemma 1', 'Lemma 2']
  refs: ['app:lemmaproofs']
  ref_targets: [{'key': 'app:lemmaproofs', 'env': 'section', 'display': None}]
  preview: k = 1, m = b) P(m = b \mid k = 1).
\]
The changes in formulas of \( F_g \) and \( F_b \) are determined by thresholds associated with their corresponding machine's signal. Specifically, the formula fo


In [19]:
CAPTION_RE = re.compile(r"\\caption\{((?:[^{}]|\{[^}]*\})*)\}", re.DOTALL)

def extract_captions(chunks, graph):
    caption_chunks = []
    for chunk in chunks:
        envs = chunk.metadata.get("chunk_environments") or []
        if "figure" not in envs and "table" not in envs:
            continue
        for m in CAPTION_RE.finditer(chunk.page_content):
            label_match = LABEL_RE.search(chunk.page_content)
            label   = label_match.group(1) if label_match else None
            display = graph["labels"].get(label, {}).get("display") if label else None
            caption_chunks.append(Document(
                page_content=f"Caption: {m.group(1).strip()}",
                metadata={
                    **chunk.metadata,
                    "is_caption":  True,
                    "caption_for": label,
                    "display":     display,
                    "displays":    [display] if display else [],
                    "chunk_id":    f"caption_{label or 'unknown'}",
                },
            ))
    return caption_chunks


In [20]:

caption_chunks = extract_captions(chunks, graph)
print(f"Extracted {len(caption_chunks)} caption chunks")
for c in caption_chunks:
    print(
        f"  {c.metadata['chunk_id']:35s} "
        f"display={c.metadata.get('display')} "
        f"preview={c.page_content[:60]}"
    )

Extracted 10 caption chunks
  caption_fig:posterior_mu_g          display=None preview=Caption: Updated belief after a positive Machine signal $(m=
  caption_fig:accuracy_comparison     display=None preview=Caption: Overall accuracy under the Aware, Unaware, and Fair
  caption_unknown                     display=None preview=Caption: 
  caption_unknown                     display=None preview=Caption: 
  caption_unknown                     display=None preview=Caption: 
  caption_unknown                     display=None preview=Caption: Minority group smaller ($\rho = 0.20$)
  caption_fig:relaxed_accuracy_a      display=None preview=Caption: $\delta_0 = 0.20$, $\delta_1 = 0.25$
  caption_fig:relaxed_accuracy_a      display=None preview=Caption: $\delta_0 = 0.20$, $\delta_1 = 0.05$
  caption_fig:relaxed_accuracy_a      display=None preview=Caption: Overall accuracy under the Aware, Unaware, and Fair
  caption_fig:relaxed_accuracy_a      display=None preview=Caption: $\delta_0 = 0.20$, $

In [23]:
PROOF_SECTION_RE = re.compile(
    r"[Pp]roof\s+of\s+"           # "Proof of" or "proof of"
    r"(?:Theorem|Proposition|Lemma|Corollary|Claim|Remark)?"
    r"\s*"
    r"(?:\\ref\{([^}]+)\}|~\\ref\{([^}]+)\})",  # \ref{key} or ~\ref{key}
    re.IGNORECASE
)

def link_proofs(chunks, graph, theorems):
    """
    Two strategies to find proof chunks, because your paper does not use
    \\begin{proof}...\\end{proof} environments. Instead proofs live under
    subsections titled 'Proof of Theorem~\\ref{...}'.

    Strategy 1 (title-based): chunk's section_title contains 'Proof of'
    and a \\ref pointing to a theorem label. This is the main signal.

    Strategy 2 (content-based): chunk lives in a section whose title
    contains 'Proof of' even if the title metadata is on a parent chunk.
    Catches spillover chunks in the same proof section.
    """
    # Build map: label_key -> chunk, for all theorem-like statement chunks
    statement_by_label = {}
    for chunk in chunks:
        envs = chunk.metadata.get("chunk_environments") or []
        if any(e in envs for e in theorems.keys()):
            for lbl in chunk.metadata.get("labels") or []:
                statement_by_label[lbl] = chunk

    print(f"{len(statement_by_label)} theorem-like statements indexed")
    print("Statement labels found:", list(statement_by_label.keys()))

    # --- Strategy 1: section_title contains "Proof of" + \ref --------
    title_linked = 0
    proof_section_titles = set()   # track which section titles are proof sections

    for chunk in chunks:
        title = chunk.metadata.get("section_title") or ""
        m = PROOF_SECTION_RE.search(title)
        if not m:
            continue

        # Extract the ref key (one of the two capture groups will have it)
        ref_key = m.group(1) or m.group(2)
        if not ref_key:
            continue

        proof_section_titles.add(title)

        if ref_key in statement_by_label:
            display = graph["labels"].get(ref_key, {}).get("display", ref_key)
            chunk.metadata["proof_of"]         = ref_key
            chunk.metadata["proof_of_display"] = display
            statement_by_label[ref_key].metadata["proved_by"] = chunk.metadata.get("chunk_id")
            title_linked += 1
        else:
            print(
                f"WARNING: proof section '{title}' references '{ref_key}' "
                f"but no statement chunk owns that label"
            )

    # --- Strategy 2: content-based fallback for chunks whose section  --
    # title is a proof section title but the ref is in the title of a   
    # *different* chunk (happens when a proof section spans many chunks) 
    content_linked = 0
    for chunk in chunks:
        if chunk.metadata.get("proof_of"):
            continue   # already linked by strategy 1
        title = chunk.metadata.get("section_title") or ""
        if title not in proof_section_titles:
            continue
        # This chunk is inside a known proof section but wasn't the
        # first chunk. Find which label this section proves by scanning
        # refs in the chunk's own content.
        for ref_key in chunk.metadata.get("refs") or []:
            if ref_key in statement_by_label:
                display = graph["labels"].get(ref_key, {}).get("display", ref_key)
                chunk.metadata["proof_of"]         = ref_key
                chunk.metadata["proof_of_display"] = display
                content_linked += 1
                break

    print(f"Linked by section title: {title_linked}")
    print(f"Linked by content fallback: {content_linked}")
    print(f"Total linked: {title_linked + content_linked}")

    # --- Spot-check: show what got linked and what did not ------------
    unlinked_statements = [
        lbl for lbl in statement_by_label
        if not statement_by_label[lbl].metadata.get("proved_by")
    ]
    if unlinked_statements:
        print(f"\nUnlinked statements ({len(unlinked_statements)}):")
        for lbl in unlinked_statements:
            display = graph["labels"].get(lbl, {}).get("display", lbl)
            print(f"  {lbl:35s} ({display})")
    else:
        print("\nAll statements have a linked proof chunk.")



In [24]:

link_proofs(chunks, graph, theorems)

# Spot-check
for chunk in chunks:
    if chunk.metadata.get("proof_of"):
        print("Proof chunk:", chunk.metadata.get("chunk_id"))
        print("  proof_of:", chunk.metadata["proof_of"])
        print("  proof_of_display:", chunk.metadata["proof_of_display"])
        print("  preview:", chunk.page_content[:150])
        break

15 theorem-like statements indexed
Statement labels found: ['app:lemma_lin', 'app:lemma_mu_relations', 'app:lemma_signal_divided', 'app:lemma_six_thresholds', 'app:lemma_indeterminate', 'sec:dms_strategy', 'prop:biased_Machine_probability', 't:unfairness_intervals', 't:fairness_condition', 'c:fair_regions_params', 't:fairness_Unaware_DM', 'c:maximum_unfairness_Unaware', 't:comparison_of_fairness', 'sec:accuracy', 't:accuracy_full']
Linked by section title: 0
Linked by content fallback: 0
Total linked: 0

Unlinked statements (15):
  app:lemma_lin                       (Lemma 1)
  app:lemma_mu_relations              (Lemma 2)
  app:lemma_signal_divided            (Lemma 3)
  app:lemma_six_thresholds            (Lemma 4)
  app:lemma_indeterminate             (Lemma 5)
  sec:dms_strategy                    (sec:dms_strategy)
  prop:biased_Machine_probability     (Proposition 1)
  t:unfairness_intervals              (Theorem 1)
  t:fairness_condition                (Theorem 2)
  c:fair_regi